# SEIRS-SEI Malaria Model

ODE approximation of SEIRS-SEI DDE model with yearly population from IBGE data, based on the results of 
### data_files/notebooks/Biting_Rate_Estimation.ipynb ### 
and
### models/seirs_sei/R0_Calculation.ipynb ###

This notebook is continued from models/seirs_sei/ode_models/Test_SEIRS_SEI_ODE.ipynb, so we will use $M'$ as fitted in the end of that notebook.

As the purpose of this test is to have $R_0=1$ for SEIRS and SEI, we will use the values of $b_1$ and $b_2$ required for that

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
DATA_DIR = '../../../data_files/data'

climate_data = pd.read_csv(DATA_DIR + '/climate_api_data_2016_2024.csv')
cases_data = pd.read_csv(DATA_DIR + '/sivep_notification_data/treated_malaria_notification_data/cumulative_manaus_cases_2016_2023.csv')
pop_data = pd.read_csv(DATA_DIR + '/ibge_manaus_population_data_2016_2024.csv')
pop_data.rename(columns={'index': 'date'}, inplace=True)

# Load IBGE population data (2000-2025) for yearly N
pop_ibge = pd.read_csv(DATA_DIR + '/ibge_manaus_rural_population_data_2000_2025.csv')
pop_by_year = dict(zip(pop_ibge['year'], pop_ibge['rural_population']))

defor_data = pd.read_csv(DATA_DIR + '/deter_notification_data/treated_deter_deforestation_data_2016_2024.csv')
fires_data = pd.read_csv(DATA_DIR + '/inpe_fire_counts_data_2016_2024.csv')

climate_data['date'] = pd.to_datetime(climate_data['date'])
cases_data['date'] = pd.to_datetime(cases_data['date'])
pop_data['date'] = pd.to_datetime(pop_data['date'])
defor_data['date'] = pd.to_datetime(defor_data['date'])
fires_data['date'] = pd.to_datetime(fires_data['date'])

# Filter data to 2017-01-01 to 2023-12-31
start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2023-12-31')
climate_data = climate_data[(climate_data['date'] >= start_date) & (climate_data['date'] <= end_date)].reset_index(drop=True)
cases_data = cases_data[(cases_data['date'] >= start_date) & (cases_data['date'] <= end_date)].reset_index(drop=True)
pop_data = pop_data[(pop_data['date'] >= start_date) & (pop_data['date'] <= end_date)].reset_index(drop=True)

print("Data loaded and filtered successfully!")
print("Climate data:", len(climate_data), "days")
print("Cases data:", len(cases_data), "days")
print("Pop data:", len(pop_data), "days")
print("Deforestation data:", len(defor_data), "days")
print("Forest fires data:", len(fires_data), "days")

print("\nIBGE rural population by year (2017-2023):")
for y in range(2017, 2024):
    print(f"{y}: {pop_by_year[y]:.2f}")

In [ ]:
#### Model Parameters 
T_prime = 16.0 
B_E = 200
p_ME = 0.9
p_ML = 0.25
p_MP = 0.75
tau_E = 1
tau_P = 1
c1 = 0.00554
c2 = -0.06737
D1 = 24.8 
#####
b1 = 0.004
b2 = 0.024
#####
A = -0.03
B = 1.31
C = -4.4
DD = 105
Tmin = 14.5  # critical min temperature for Plasmodium development
gamma = 1/120
R_L = 32.67

# SEIRS parameters
tau_H = 10
omega = 1/365

# Critical temperatures for Plasmodium development
optimal_temp_plasm = 23.5
critical_max_temp_plasm = 29.8
#critical_min_temp_plasm = 14.5 # Tmin

# Critical temperatures for Anopheles development
optimal_temp_anop = 25 #26
critical_max_temp_anop = 34 #40
critical_min_temp_anop = 16 #10

In [ ]:
### Helper functions using passed parameters
def tau_L(Temp):
    denom = c1 * Temp + c2
    return 1.0/denom if denom > 0 else 100.0

def p_T(Temp, Humid, H0=58.0, k=0.25, phi=0.05):
    """Adult survival probability of Anopheles darlingi in the Amazon (Epidbot)
    
    Input:
      H0 = 58: Midpoint of the sigmoid (humidity where p_H ≈ 0.5)
      k = 0.25: Transition slope
      phi = 0.05: Minimum survival (micro-habitats)
    
    NOTE: This should use umid_min (daily minimum), NOT umid_med.
    """
    
    denominator = A * Temp**2 + B * Temp + C
    p_T_temp = np.where(denominator <= 0, 0.0, np.exp(-1 / denominator))
    
    p_H = phi + (1.0 - phi) / (1.0 + np.exp(-k * (Humid - H0)))
    
    return p_T_temp * p_H

def mu(Temp, Humid, H0=58.0, k=0.25, phi=0.05):
    p_val = p_T(Temp, Humid, H0, k, phi)
    return -np.log(p_val) if p_val > 0 else 1.0

def a(Temp):
    return max(0, (Temp - T_prime)/D1)

def tau_M(Temp):
    diff = Temp - Tmin
    return DD/diff if diff > 0 else 1000.0

def b_rate(Rain, Temp):
    tau_L_curr = tau_L(Temp)
    if tau_L_curr <= 0: return 0.0
    p_ER = (4*p_ME/R_L**2)*Rain*(R_L - Rain) if 0<=Rain<=R_L else 0.0
    p_LR = (4*p_ML/R_L**2)*Rain*(R_L - Rain) if 0<=Rain<=R_L else 0.0
    p_LT = np.exp(-(c1*Temp + c2))
    p_PR = (4*p_MP/R_L**2)*Rain*(R_L - Rain) if 0<=Rain<=R_L else 0.0
    numer = B_E * p_ER * p_LR * p_LT * p_PR
    denom = tau_E + tau_L_curr + tau_P
    return numer/denom if denom > 0 else 0.0

def b3_briere_unscaled_plasm(Temp):
    return Temp * (Temp - Tmin) * (critical_max_temp_plasm - Temp)**(1/2)
    
unscaled_peak_value_plasm = b3_briere_unscaled_plasm(optimal_temp_plasm)

desired_peak_rate_plasm = 1.0 / tau_M(optimal_temp_plasm) ### b3 at the optimal temperature
eta_plasm = desired_peak_rate_plasm / unscaled_peak_value_plasm

def b3_briere_scaled_plasm(Temp):
    if Temp <= Tmin or Temp >= critical_max_temp_plasm:
        return 0.0
    return eta_plasm * Temp * (Temp - Tmin) * ((critical_max_temp_plasm - Temp)**(1/2))

def temp_factor_briere_normalized_anop(Temp): #f_T(T)
    if Temp <= critical_min_temp_anop or Temp >= critical_max_temp_anop:
        return 0.0
    raw = Temp * (Temp - critical_min_temp_anop) * ((critical_max_temp_anop - Temp) ** 0.5)
    opt_raw = optimal_temp_anop * (optimal_temp_anop - critical_min_temp_anop) * ((critical_max_temp_anop - optimal_temp_anop) ** 0.5)
    return min(1.0, raw / opt_raw)

In [ ]:
### E/I, in the steady state, when dI_M/dt=0, it follows that b_3\ell E = mu I --> E/I = mu/(b_3\ell) 
### = mu/((1/tau_M)/f(T) * exp(-mu * tau_M)), wehere f(T) is the Briere ratio
### = mu*tau_M*exp(mu * tau_M)/f(T)

def exposed_to_infected_ratio(Temp, Humid, H0=58.0, k=0.25, phi=0.05):
    return (
        mu(Temp, Humid, H0=58.0, k=0.25, phi=0.05) * 
        tau_M(Temp) *
        np.exp(mu(Temp, Humid, H0=58.0, k=0.25, phi=0.05) * tau_M(Temp)) /
        (b3_briere_unscaled_plasm(Temp)/unscaled_peak_value_plasm)
           )

temp_med_0 = climate_data['temp_med'][0]
umid_min_0 = climate_data['umid_min'][0]

initial_exposed_to_infected_ratio = exposed_to_infected_ratio(temp_med_0, umid_min_0)
print(f'The initial ratio of exposed to infected mosquitos was estimated to be ~{round(initial_exposed_to_infected_ratio)}')

In [ ]:
# Rural cases for initial conditions
rural_cases_df = cases_data.copy()
rural_cases_df[['active_total', 'active_symptomatic', 'active_asymptomatic', 'new_cases', 'cumulative_cases', 'active_per_new_case']] = rural_cases_df[['active_total', 'active_symptomatic', 'active_asymptomatic', 'new_cases', 'cumulative_cases', 'active_per_new_case']]*(41.54/100)
I_H0 = round(rural_cases_df['active_total'].iloc[0])
### We will estimate that E_H0 represents about 5% of the human population, while R_H0 is ~15%, given the acquired immunity over time
N_2017 = round(pop_by_year[2017])

E_H0 = round(N_2017 * 0.05)
R_H0 = round(N_2017 * 0.15)

S_H0 = N_2017 - E_H0 - I_H0 - R_H0
### We will later optimize S_H0 to best fit our data

### For starters, we will estimate that the typical ratio of vectors to hosts is in the range of 10-100.
### In the case of Manaus, especially in the rural area, which is a peri-urban area with high humidity, 
### we could put this range at 20-50, so let's say 35:
M_0 = round(9.9 * N_2017)
### We will later optimize M (M_0) to best fit our data
### We will also estimate that I_M0 represents ~1% of the initial mosquito population, while E_M0 represents 3 I_M0, as estimated above.
I_M0 = round(M_0 * 0.01)
E_M0 = round(I_M0 * 3)
S_M0 = M_0 - E_M0 - I_M0
initial_state_2017 = np.array([S_H0, E_H0, I_H0, R_H0, S_M0, E_M0, I_M0])

print(f"Initial human compartments in 2017: N={N_2017}, S_H={S_H0}, E_H={E_H0}, I_H={I_H0}, R_H={R_H0}")
print(f"Initial mosquito compartments in 2017: M={M_0}, S_M={S_M0}, E_M={E_M0}, I_M={I_M0}")

M_prime = 494121

In [ ]:
def seirs_sei_ode(t, z, N, year_climate,
                    T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P,
                    c1, c2, D1, b1, b2, A, B, C, DD, 
                    Tmin, optimal_temp_plasm, critical_max_temp_plasm,
                    optimal_temp_anop, critical_max_temp_anop, critical_min_temp_anop,
                    gamma, R_L, M_prime, tau_H, omega, 
                    b3_h=None, b3_m=None):
    
    """SEIRS-SEI ODE with delay approximation (b3 parameters)
    
    Parameters:
    - t: time
    - z: state vector [S_H, E_H, I_H, R_H, S_M, E_M, I_M]
    - N: human population (constant for the year)
    - year_climate: climate DataFrame for the year
    - Model parameters: T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P
    - Climate params: c1, c2, D1, b1, b2, A, B, C, DD, Tmin
    - Epi params: gamma, R_L, M, tau_H, omega
    - b3_h: human delay approx (default 1/tau_H)
    - b3_m: mosquito delay approx (calculated from temp if None)
    """
    # Get climate data
    day_idx = int(t)
    day_idx = np.clip(day_idx, 0, len(year_climate)-1)
    T_curr = year_climate.iloc[day_idx]['temp_med']
    R_curr = year_climate.iloc[day_idx]['precip_med']
    H_curr = year_climate.iloc[day_idx]['umid_min'] ### Substitutes umid_med for umid_min, as that is the humidity that causes stress on the vector
    
    # Current parameters
    mu_curr = mu(T_curr, H_curr)
    a_curr = a(T_curr)
    tau_M_curr = tau_M(T_curr)

    # Delay approximation (1/tau_M & 1/tau_H):
    # Use provided b3_m or calculate from tau_M
    if b3_m is None:
        b3_m_curr = 1.0/tau_M_curr if tau_M_curr > 0 else 0.0
    else:
        b3_m_curr = b3_m
    
    # Use provided b3_h or calculate from tau_H
    if b3_h is None:
        b3_h_curr = 1.0/tau_H if tau_H > 0 else 0.0
    else:
        b3_h_curr = b3_h
    
    l_curr = np.exp(-mu_curr * tau_M_curr)
    b_curr = b_rate(R_curr, T_curr)

    b3_briere_curr = b3_briere_scaled_plasm(T_curr) #### b3, the transition rate from E to I, is the development rate of the Plasmodium
    
    # States
    S_H, E_H, I_H, R_H, S_M, E_M, I_M = z
    
    # Forces of infection
    foi_h = a_curr * b2 * (I_M / N)
    foi_m = a_curr * b1 * (I_H / N)
    
    # Human ODEs (SEIRS)
    dShdt = -foi_h * S_H + omega * R_H
    dEhdt = foi_h * S_H - b3_h_curr * E_H
    dIhdt = b3_h_curr * E_H - gamma * I_H
    dRhdt = gamma * I_H - omega * R_H
    
    # Mosquito ODEs (SEI)

    temp_factor = temp_factor_briere_normalized_anop(T_curr)
    
    habitat_creating_factor = 2*R_curr/R_L
    habitat_flushing_factor = np.exp(1 - (2*R_curr/R_L))
    rain_factor = habitat_creating_factor*habitat_flushing_factor # min(1, max(0, R_curr/30))
    
    K = M_prime * temp_factor * rain_factor
    total_mosq = S_M + E_M + I_M

    ### We decided to include a modification of the Verhurst model: dM/dt = bM(1-M/K):
    ### In this case, if the population of mosquitos ever reaches 0, the growth term becomes 0,
    ### and the population is finished. 
    ### Changing it for dM/dt = bK(1-M/K), we consider that the recruitment comes from a persistent
    ### presence of larvae and eggs in the environment. Even without adult mosquitos, with M approaching 0
    ### the system gives a recovery of the population at a maximum rate of K, making the model more resilient
    ### to environmental shocks.
    density_factor = max(0, 1 - total_mosq/K) if K>0 else 0.0
    mosquito_birth = b_curr * density_factor * K
    ### If M is too small, (1-M/K) is close to 1, so the population grows rapidly

    
    dSmdt = mosquito_birth - foi_m * S_M - mu_curr * S_M
    dEmdt = foi_m * S_M - (mu_curr + (b3_briere_curr * l_curr)) * E_M #foi_m * S_M - (mu_curr + (b3_m_curr * l_curr)) * E_M
    dImdt = b3_briere_curr * l_curr * E_M  - mu_curr * I_M #b3_m_curr * l_curr * E_M  - mu_curr * I_M
    
    # Prevent negative derivatives
    for d in [dShdt, dEhdt, dIhdt, dRhdt, dSmdt, dEmdt, dImdt]:
        d = max(d, -1e6)
    
    return [dShdt, dEhdt, dIhdt, dRhdt, dSmdt, dEmdt, dImdt]


In [ ]:
#### Yearly Simulation (2017-2023)
years = list(range(2017, 2024))
all_results = []
current_state = initial_state_2017.copy()

for year in years:
    print(f"\nSimulating {year}...")
    N = round(pop_by_year[year])
    start = f"{year}-01-01"
    end = f"{year}-12-31"
    year_climate = climate_data[(climate_data['date'] >= start) & (climate_data['date'] <= end)].reset_index(drop=True)
    num_days = len(year_climate)
    print(f"Year {year}: N={N}, Days={num_days}")
    
    def ode_func(t, z):
        return seirs_sei_ode(t, z, N, year_climate,
                            T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P,
                            c1, c2, D1, b1, b2, A, B, C, DD, 
                            Tmin, optimal_temp_plasm, critical_max_temp_plasm,
                            optimal_temp_anop, critical_max_temp_anop, critical_min_temp_anop,
                            gamma, R_L, M_prime, tau_H, omega)
    
    sol = solve_ivp(ode_func, [0, num_days], current_state, 
                    t_eval=np.linspace(0, num_days, num_days*10), method='LSODA') #'DOP853')
    
    all_results.append({'year': year, 'N': N, 'sol': sol, 'climate': year_climate})
    current_state = sol.y[:, -1].copy()
    print(f"Finished {year}. Final I_H: {round(current_state[2])}, Final I_M: {round(current_state[6])}")

print("\nAll years simulated successfully!")

In [ ]:
#### Combine and Plot Results
combined_df = pd.DataFrame()
for res in all_results:
    year = res['year']
    sol = res['sol']
    climate = res['climate']
    
    # Interpolate to daily resolution
    t_interp = np.linspace(0, sol.t[-1], len(climate))
    interp = [interp1d(sol.t, sol.y[i])(t_interp) for i in range(7)]
    
    df = pd.DataFrame({
        'date': pd.to_datetime(climate['date']).values,
        'S_H': interp[0], 'E_H': interp[1], 'I_H': interp[2], 'R_H': interp[3],
        'S_M': interp[4], 'E_M': interp[5], 'I_M': interp[6], 'N': res['N']
    })
    combined_df = pd.concat([combined_df, df]).reset_index(drop=True)

# Ensure date columns are datetime
combined_df['date'] = pd.to_datetime(combined_df['date'])
rural_cases_df['date'] = pd.to_datetime(rural_cases_df['date'])

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0,0].plot(combined_df['date'], combined_df['I_H'], 'b-', linewidth=2, label='Infectious Humans', alpha=0.8)
axes[0,0].plot(rural_cases_df['date'], rural_cases_df['active_total'], 'r-', linewidth=1.5, 
        label='Observed Active Rural Cases', alpha=0.7, marker='o', 
        markersize=3, markevery=30)  # Show markers every ~1 month for readability
axes[0,0].set_title('Model vs Data: Infectious Humans')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(combined_df['date'], combined_df['S_H'], label='Susceptible')
axes[0,1].plot(combined_df['date'], combined_df['E_H'], label='Exposed', linestyle='--')
axes[0,1].plot(combined_df['date'], combined_df['I_H'], label='Infected', linestyle='-')
axes[0,1].plot(combined_df['date'], combined_df['R_H'], label='Recovered', linestyle=':')
axes[0,1].set_title('Human Compartments'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(combined_df['date'], combined_df['S_M'], label='Susceptible', linewidth=1.5)
axes[1,0].plot(combined_df['date'], combined_df['E_M'], label='Exposed', linewidth=1.5, linestyle='--')
axes[1,0].plot(combined_df['date'], combined_df['I_M'], label='Infectious', linewidth=2)
axes[1,0].set_title('Mosquito Compartments'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(combined_df['date'], combined_df['E_M'], label='Exposed', linewidth=1.5, linestyle='--')
axes[1,1].plot(combined_df['date'], combined_df['I_M'], label='Infectious', linewidth=2)
axes[1,1].set_title('Mosquito Compartments - Exposed and Infected'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Infectious humans vs observed data
axes[0,0].plot(
    combined_df['date'],
    combined_df['I_H'],
    'b-',
    linewidth=2,
    label='Infectious Humans',
    alpha=0.8
)

axes[0,0].plot(
    rural_cases_df['date'],
    rural_cases_df['active_total'],
    'r-',
    linewidth=1.5,
    label='Observed Active Rural Cases',
    alpha=0.7,
    marker='o',
    markersize=3,
    markevery=30
)

axes[0,0].set_title('Model vs Data: Infectious Humans')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)


# Human compartments
axes[0,1].plot(combined_df['date'], combined_df['S_H'], label='Susceptible')
axes[0,1].plot(combined_df['date'], combined_df['E_H'], label='Exposed', linestyle='--')
axes[0,1].plot(combined_df['date'], combined_df['I_H'], label='Infected', linestyle='-')
axes[0,1].plot(combined_df['date'], combined_df['R_H'], label='Recovered', linestyle=':')

axes[0,1].set_title('Human Compartments')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)


# Mosquito compartments (LOG SCALE)
axes[1,0].plot(
    combined_df['date'],
    combined_df['S_M'],
    label='Susceptible',
    linewidth=1.5
)

axes[1,0].plot(
    combined_df['date'],
    combined_df['E_M'],
    label='Exposed',
    linewidth=1.5,
    linestyle='--'
)

axes[1,0].plot(
    combined_df['date'],
    combined_df['I_M'],
    label='Infectious',
    linewidth=2
)

# Apply logarithmic scale
axes[1,0].set_yscale('log')
axes[1,0].set_title('Mosquito Compartments (Log Scale)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3, which='both')

axes[1,1].plot(
    combined_df['date'],
    combined_df['E_M'],
    label='Exposed',
    linewidth=1.5,
    linestyle='--'
)

axes[1,1].plot(
    combined_df['date'],
    combined_df['I_M'],
    label='Infectious',
    linewidth=2
)

# Apply logarithmic scale
axes[1,1].set_yscale('log')
axes[1,1].set_title('Mosquito Compartments (Log Scale)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
combined_df['Temp'] = climate_data['temp_med'].values[:len(combined_df)]

# Compute a(T) and b3(T)
combined_df['a_T'] = combined_df['Temp'].apply(a)
combined_df['b_3'] = combined_df['Temp'].apply(b3_briere_scaled_plasm)

# Infectious mosquito ratio
combined_df['IM_over_N'] = combined_df['I_M'] / combined_df['N']

# Human force of infection
combined_df['lambda_H'] = ((b2/combined_df['b_3'][0]) * 
    combined_df['a_T']
    * combined_df['IM_over_N']
    * combined_df['S_H'][0]/combined_df['E_H'][0]                       
)

# ============================================================
# Create grid for contourf
# ============================================================

x = combined_df['IM_over_N'].values
y = combined_df['a_T'].values
z = combined_df['lambda_H'].values

# Grid definition
xi = np.linspace(x.min(), x.max(), 200)
yi = np.linspace(y.min(), y.max(), 200)

Xi, Yi = np.meshgrid(xi, yi)

# Interpolate z values onto the grid
from scipy.interpolate import griddata

Zi = griddata(
    (x, y),
    z,
    (Xi, Yi),
    method='linear'
)

# ============================================================
# Plot contourf heatmap
# ============================================================

plt.figure(figsize=(10, 7))

contour = plt.contourf(
    Xi,
    Yi,
    Zi,
    levels=25,
    cmap='viridis'
)

critical_line = plt.contour(
    Xi,
    Yi,
    Zi,
    levels=[1],
    colors='red',
    linewidths=2.5
)

plt.clabel(
    critical_line,
    fmt={1: r'$\lambda_H = 1$'},
    fontsize=11
)

# Colorbar
cbar = plt.colorbar(contour)
cbar.set_label(r'Human Force of Infection $\lambda_H$', fontsize=12)

# Labels
plt.xlabel(r'Infectious Mosquito Ratio $I_M/N$', fontsize=12)
plt.ylabel(r'Biting Rate $a(T)$', fontsize=12)

plt.title(
    r'Heatmap of Human Force of Infection',
    fontsize=14
)

plt.tight_layout()
plt.show()

### _____________________________________________________________________________________________________________________ ###

When $R_0=1$, it is clear that the disease does not stabilize and simulate the expected behavior. As so, we decided instead to fit values of $b_1, \ b_2$ in order to minimize the RMSE between the number of infected according to the model and the cases data, as was done previously with $M'$

In [ ]:
def run_simulation_with_b1b2(b1_val, b2_val, initial_state=None):
    if initial_state is None:
        init_state = initial_state_2017.copy()
    else:
        init_state = initial_state.copy()
    all_IH = []
    all_dates = []
    current_state = init_state
    for year in years:
        N = round(pop_by_year[year])
        start = f"{year}-01-01"
        end = f"{year}-12-31"
        year_climate = climate_data[(climate_data["date"] >= start) & (climate_data["date"] <= end)].reset_index(drop=True)
        num_days = len(year_climate)
        def ode_func(t, z):
            return seirs_sei_ode(t, z, N, year_climate,
                                T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P,
                                c1, c2, D1, b1_val, b2_val, A, B, C, DD,
                                Tmin, optimal_temp_plasm, critical_max_temp_plasm,
                                optimal_temp_anop, critical_max_temp_anop, critical_min_temp_anop,
                                gamma, R_L, M_prime, tau_H, omega)
        sol = solve_ivp(ode_func, [0, num_days], current_state,
                        t_eval=np.linspace(0, num_days, num_days*10), method="LSODA")
        if not sol.success:
            return np.array([]), np.array([])
        current_state = sol.y[:, -1].copy()
        current_state = np.maximum(current_state, 0)
        t_interp = np.linspace(0, sol.t[-1], len(year_climate))
        IH_interp = interp1d(sol.t, sol.y[2])(t_interp)
        all_IH.extend(IH_interp)
        all_dates.extend(year_climate["date"].values)
    return np.array(all_dates), np.array(all_IH)

In [ ]:
def objective_rmse(params):
    b1_val, b2_val = params
    if b1_val <= 0 or b1_val > 1 or b2_val <= 0 or b2_val > 1:
        return 1e15
    dates, IH = run_simulation_with_b1b2(b1_val, b2_val)
    if len(dates) == 0:
        return 1e15
    model_interp = interp1d(pd.to_datetime(dates).astype(np.int64), IH, kind="linear", fill_value="extrapolate")
    obs_dates_int = rural_cases_df["date"].astype(np.int64).values
    model_at_obs = model_interp(obs_dates_int)
    observed = rural_cases_df["active_total"].values
    rmse = np.sqrt(np.mean((model_at_obs - observed) ** 2))
    return rmse

In [ ]:
print("Grid search for optimal (b1, b2)...")
# b1_candidates = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
# b2_candidates = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]

### In order to reduce run time, we will use the best results from the first execution, and some other values just to check
### Best from grid: b1=0.2000, b2=0.0200 (RMSE=537.52)

b1_candidates = [0.01, 0.04, 0.2, 0.5, 0.8]
b2_candidates = [0.02, 0.09, 0.2, 0.5, 0.8]

grid_results = []
n_grid = len(b1_candidates) * len(b2_candidates)
count = 0
for b1v in b1_candidates:
    for b2v in b2_candidates:
        count += 1
        rmse = objective_rmse([b1v, b2v])
        grid_results.append((b1v, b2v, rmse))
        print(f"  [{count}/{n_grid}] b1={b1v:.4f}, b2={b2v:.4f} -> RMSE={rmse:.2f}")
grid_results.sort(key=lambda x: x[2])
best_b1, best_b2, best_rmse = grid_results[0]
print(f"\nBest from grid: b1={best_b1:.4f}, b2={best_b2:.4f} (RMSE={best_rmse:.2f})")

In [ ]:
print("Local optimization around best grid point...")
result = minimize(
    objective_rmse,
    x0=[best_b1, best_b2],
    method="Nelder-Mead",
    options={"xatol": 1e-4, "fatol": 0.1, "maxiter": 50}
)
b1_opt, b2_opt = result.x
rmse_opt = result.fun
print(f"Optimized: b1={b1_opt:.6f}, b2={b2_opt:.6f} (RMSE={rmse_opt:.2f})")

In [ ]:
print("Final simulation with optimal (b1, b2)...")
dates_opt, IH_opt = run_simulation_with_b1b2(b1_opt, b2_opt)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

ax1 = axes[0]
b1_vals = sorted(set(r[0] for r in grid_results))
b2_vals = sorted(set(r[1] for r in grid_results))
rmse_mat = np.full((len(b1_vals), len(b2_vals)), np.nan)
for b1v, b2v, r in grid_results:
    i = b1_vals.index(b1v)
    j = b2_vals.index(b2v)
    rmse_mat[i, j] = r
X, Y = np.meshgrid(b2_vals, b1_vals)
contour = ax1.contourf(X, Y, rmse_mat, levels=20, cmap='viridis')
cbar = plt.colorbar(contour, ax=ax1, label='RMSE')
ax1.plot(b2_opt, b1_opt, 'r*', markersize=15,
         label=f'Optimum\nb1={b1_opt:.4f}\nb2={b2_opt:.4f}')
ax1.plot(best_b2, best_b1, 'wo', markersize=8,
         label=f'Best grid\nb1={best_b1:.4f}\nb2={best_b2:.4f}')
ax1.set_xlabel('b2')
ax1.set_ylabel('b1')
ax1.set_title('RMSE contour over (b1, b2) grid')
ax1.legend()

ax2 = axes[1]
ax2.plot(pd.to_datetime(dates_opt), IH_opt, color='steelblue',
         linewidth=1.5,
         label=f'Model I_H\nb1={b1_opt:.4f}, b2={b2_opt:.4f}')
ax2.plot(rural_cases_df['date'], rural_cases_df['active_total'],
         color='red', linewidth=1.0, alpha=0.7,
         label='Observed (active_total)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Active cases')
ax2.set_title('Prevalence: Model vs Observed (Manaus 2017-2023)')
ax2.legend()

plt.tight_layout()
#plt.savefig('calibracao_b1b2.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCalibration Summary")
print(f"  M_prime fixed:               {M_prime:>10,.0f}")
print(f"  Initial b1 (R0=1):            {0.004:.4f}")
print(f"  Initial b2 (R0=1):            {0.024:.4f}")
print(f"  Optimized b1:                 {b1_opt:.6f}")
print(f"  Optimized b2:                 {b2_opt:.6f}")
print(f"  Final RMSE:                   {rmse_opt:.2f}")